From `part-2.ipynb`, we fit class-specific period-luminosity and period-color relations for
both RRab and RRc stars using a calibration sample ($N_\mathrm{RRab} = 559$, $N_\mathrm{RRc} = 315$).  Here
we apply the class-specific period-color relations to the full Gaia DR3 RR
Lyrae catalog including low-latitude and low-parallax-SNR stars that were
excluded before to derive the color excess $E(G_\mathrm{BP}-G_\mathrm{RP})$
for each star and build a full-sky reddening map.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ugdatalab import get_gaia, rrlyrae_class_mask, rrlyrae_representative_period

PC_FILES = {
    'RRab': Path('../archive/flat_pc_rrab.npy'),
    'RRc': Path('../archive/flat_pc_rrc.npy'),
}

pc_posteriors = {rr_class: np.load(path) for rr_class, path in PC_FILES.items()}
pc_summary = {}
for rr_class, flat in pc_posteriors.items():
    pc_summary[rr_class] = {
        'slope_median': np.median(flat[:, 0]),
        'slope_std': np.std(flat[:, 0]),
        'intercept_median': np.median(flat[:, 1]),
        'intercept_std': np.std(flat[:, 1]),
        'intrinsic_sigma_median': 10 ** np.median(flat[:, 2]),
        'intrinsic_sigma_std': np.std(10 ** flat[:, 2]),
    }
    print(
        f"{rr_class} period-color relation: "
        f"a_c={pc_summary[rr_class]['slope_median']:.4f} +/- {pc_summary[rr_class]['slope_std']:.4f}, "
        f"b_c={pc_summary[rr_class]['intercept_median']:.4f} +/- {pc_summary[rr_class]['intercept_std']:.4f}, "
        f"sigma_c={pc_summary[rr_class]['intrinsic_sigma_median']:.4f} mag"
    )


# 24, 25 & 26

### Calculate the color excess, $E\left(G_\mathrm{BP} − G_\mathrm{RP}\right) = \left(G_\mathrm{BP} − G_\mathrm{RP}\right)_\mathrm{observed} − \left(G_\mathrm{BP} − G_\mathrm{RP}\right)_\mathrm{intrinsic}$, for all RR Lyraes in the catalog. From this, calculate $A_\mathrm{G}$, the G-band extinction, for each star. You may assume $R_\mathrm{G} \equiv \frac{A_\mathrm{G}}{E\left(G_\mathrm{BP} − G_\mathrm{RP}\right)} = 2.0$.

## Compare your calculated $A_\mathrm{G}$ values to the `G_absorption` value provided in the RR Lyrae catalog.

We propagate three uncertainty terms in quadrature:
1. observational color noise from BP/RP photometric SNR,
2. coefficient uncertainty from the posterior spread in slope and intercept,
3. intrinsic color scatter from the fitted $\sigma_c$ for each respective RR Lyrae class.


In [ ]:
query_full = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
"""

raw_full = get_gaia(query_full)
period = rrlyrae_representative_period(raw_full)

class_masks = {
    'RRab': rrlyrae_class_mask(raw_full, 'RRab'),
    'RRc': rrlyrae_class_mask(raw_full, 'RRc'),
}
rrd_mask = rrlyrae_class_mask(raw_full, 'RRd')

print(f"Full catalog: {len(raw_full):,} stars")
print(f"  RRab: {class_masks['RRab'].sum():,}")
print(f"  RRc:  {class_masks['RRc'].sum():,}")
print(f"  RRd:  {rrd_mask.sum():,}")


In [ ]:
l_all = raw_full['l']
b_all = raw_full['b']
l_plot = np.where(l_all > 180, l_all - 360, l_all)

fig = plt.figure(figsize=(12, 6), dpi=300)
ax = fig.add_subplot(111, projection='mollweide')

ax.scatter(
    np.deg2rad(l_plot[class_masks['RRab']]),
    np.deg2rad(b_all[class_masks['RRab']]),
    s=0.3,
    alpha=0.3,
    color='C0',
    rasterized=True,
    label=f"RRab ({class_masks['RRab'].sum():,})",
)
ax.scatter(
    np.deg2rad(l_plot[class_masks['RRc']]),
    np.deg2rad(b_all[class_masks['RRc']]),
    s=0.3,
    alpha=0.3,
    color='C1',
    rasterized=True,
    label=f"RRc ({class_masks['RRc'].sum():,})",
)
ax.scatter(
    np.deg2rad(l_plot[rrd_mask]),
    np.deg2rad(b_all[rrd_mask]),
    s=0.3,
    alpha=0.3,
    color='C2',
    rasterized=True,
    label=f"RRd ({rrd_mask.sum():,})",
)

ax.set_title(
    f"Full Gaia DR3 RR Lyrae sky distribution "
    f"(RRab={class_masks['RRab'].sum()}, RRc={class_masks['RRc'].sum()}, RRd={rrd_mask.sum()})"
)
ax.set_xlabel('Galactic longitude l')
ax.set_ylabel('Galactic latitude b')
ax.legend(loc='upper right', markerscale=6, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
log10_P = np.log10(period)
color_obs = raw_full["bp_rp"]

snr_bp = raw_full["phot_bp_mean_flux_over_error"]
snr_rp = raw_full["phot_rp_mean_flux_over_error"]
sig_obs = (2.5 / np.log(10)) * np.sqrt(1.0 / snr_bp**2 + 1.0 / snr_rp**2)

color_int = np.full(len(raw_full), np.nan)
sigma_coeff = np.full(len(raw_full), np.nan)
sigma_intrinsic = np.full(len(raw_full), np.nan)

for rr_class, mask in class_masks.items():
    summary = pc_summary[rr_class]
    color_int[mask] = (
        summary['slope_median'] * log10_P[mask] + summary['intercept_median']
    )
    sigma_coeff[mask] = np.sqrt(
        (summary['slope_std'] * log10_P[mask])**2 + summary['intercept_std']**2
    )
    sigma_intrinsic[mask] = summary['intrinsic_sigma_median']

E_bprp = color_obs - color_int
A_G_calc = 2.0 * E_bprp
sigma_E = np.sqrt(sig_obs**2 + sigma_coeff**2 + sigma_intrinsic**2)

raw_full["E_bprp"] = E_bprp
raw_full["A_G_calc"] = A_G_calc
raw_full["sigma_E"] = sigma_E
raw_full["color_int"] = color_int

valid = np.isfinite(E_bprp)
print(f"Stars with valid class-specific E(BP-RP): {valid.sum():,}")
for rr_class, mask in class_masks.items():
    class_valid = valid & mask
    print(
        f"  {rr_class}: {class_valid.sum():,} stars"
    )
print(f"  RRd:  {rrd_mask.sum():,} stars (kept separate; no single-mode RRd relation applied)")


# 26

The `gaiadr3.vari_rrlyrae` table includes a `g_absorption` column computed by
the Gaia pipeline from the SFD dust map.  Comparing that catalog quantity to our
class-specific empirical $A_G$ provides a useful sanity check: both trace G-band
extinction, but through independent constructions.

In [ ]:
g_abs_cat = raw_full['g_absorption']
A_G_emp = raw_full['A_G_calc']

ok26 = np.isfinite(g_abs_cat) & np.isfinite(A_G_emp)
resid = A_G_emp[ok26] - g_abs_cat[ok26]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300)

ax0 = axes[0]
ax0.scatter(g_abs_cat[ok26], A_G_emp[ok26], s=0.5, alpha=0.15, color='C0', rasterized=True)
lim = (0, 5)
ax0.plot(lim, lim, 'k--', lw=1.0, label='1:1')
ax0.set_xlim(lim)
ax0.set_ylim(lim)
ax0.set_xlabel(r"$A_G$ (Gaia catalog, SFD-based) [mag]")
ax0.set_ylabel(r"$A_G$ (empirical, class-specific period-color) [mag]")
ax0.set_title('Catalog vs. empirical G-band extinction')
ax0.legend(fontsize=9)
ax0.grid(True, alpha=0.3)

ax1 = axes[1]
ax1.hist(resid, bins=100, range=(-3, 3), color='C1', alpha=0.7, density=True)
med_r = np.median(resid)
ax1.axvline(med_r, color='k', lw=1.5, linestyle='--', label=f'median = {med_r:.3f}')
ax1.set_xlabel(r"$A_G^\mathrm{empirical} - A_G^\mathrm{catalog}$ [mag]")
ax1.set_ylabel('Density')
ax1.set_title('Residual distribution')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


At low extinction ($A_G \lesssim 1$ mag) the
class-specific empirical values track the SFD-based catalog values reasonably
well.  The largest residuals still cluster near the Galactic plane, where the
SFD total-column map, crowding, and unresolved blending all become more severe.

# Highlight separately for RRab and RRc
# Color code by |b| perhaps to show latitude-dependent variance

# 27


In [ ]:
l_v = np.where(l_all > 180, l_all - 360, l_all)
fin_v = np.isfinite(E_bprp)

fig = plt.figure(figsize=(12, 6), dpi=300)
ax = fig.add_subplot(111, projection='mollweide')
ax.set_facecolor('black')
sc = ax.scatter(
    np.deg2rad(l_v[fin_v]),
    np.deg2rad(b_all[fin_v]),
    c=E_bprp[fin_v],
    cmap='afmhot',
    vmin=0,
    vmax=2,
    s=0.3,
    alpha=0.4,
    rasterized=True,
)
plt.colorbar(sc, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8,
             label=r"$E(G_\mathrm{BP} - G_\mathrm{RP})$ [mag]")
ax.set_title(f"Uncut reddening map")
ax.grid(True, alpha=0.3, color='white')
plt.tight_layout()
plt.show()
